# Unidad 2: Programación Orientada a Objetos (POO)
**Materia:** Programación — Licenciatura en Negocios Digitales (3º año)
**Institución:** Universidad del Museo Social Argentino (UMSA)

---

## Introducción y Contexto de Negocio

A diferencia de los scripts de análisis de datos que son de naturaleza lineal, las plataformas SaaS, las fintech o los portales de e-commerce son sistemas complejos compuestos de entidades que interactúan de forma continua.

La **Programación Orientada a Objetos (POO)** es el paradigma fundamental de desarrollo que nos permite modelar el negocio real en código. Unificando **datos** (atributos) y **comportamiento** (métodos), podemos crear piezas de código modulares, testeables y altamente reutilizables. En esta unidad, aprenderemos a modelar usuarios, carritos, productos y transacciones reales del backend de un negocio digital.

### Objetivos de Aprendizaje:
1. Modelar clases de negocio usando atributos de instancia, de clase y constructores.
2. Implementar los pilares de Abstracción y Encapsulamiento usando atributos privados y el decorador `@property`.
3. Aplicar los conceptos de Herencia y Polimorfismo en el diseño de componentes flexibles (suscripciones, pasarelas de pago).
4. Diseñar interacciones complejas entre múltiples objetos (Usuario -> Carrito -> Productos).


## 1. Clases, Objetos y Atributos

Una **Clase** es la plantilla o molde que describe los atributos y métodos de una entidad. Un **Objeto** es una instancia física creada a partir de esa plantilla.


In [ ]:
class Usuario:
    # Atributo de Clase (compartido por todas las instancias)
    plataforma = "UMSA Digital Hub"

    def __init__(self, nombre: str, email: str):
        # Atributos de Instancia (propios de cada objeto)
        self.nombre = nombre
        self.email = email
        self.activo = True

    # Métodos Especiales (Dunder Methods)
    def __str__(self):
        # Representación amigable para humanos
        return f"{self.nombre} ({self.email})"

    def __repr__(self):
        # Representación técnica para desarrolladores
        return f"Usuario(nombre='{self.nombre}', email='{self.email}')"

    # Método regular
    def suspender(self):
        self.activo = False
        print(f"El usuario {self.nombre} ha sido suspendido temporalmente.")

# Creación de instancias
usuario_1 = Usuario("Paula Ferreyra", "paula@umsa.edu.ar")
usuario_2 = Usuario("Martín Gómez", "martin@umsa.edu.ar")

print("Representación str:", str(usuario_1))
print("Representación repr:", repr(usuario_2))

usuario_2.suspender()
print(f"¿Usuario 2 activo?: {usuario_2.activo}")


## 2. Abstracción y Encapsulamiento con `@property`

El **Encapsulamiento** consiste en ocultar los detalles internos de un objeto y proteger su estado, impidiendo modificaciones directas inválidas. 
En Python, marcamos los atributos privados usando doble guión bajo (`__`) y controlamos su lectura y modificación por medio de **getters** y **setters** utilizando el decorador `@property`.


In [ ]:
class Producto:
    def __init__(self, nombre: str, precio_base: float):
        self.nombre = nombre
        # Usamos el setter internamente para aplicar la validación inicial
        self.precio = precio_base

    # Getter: expone el valor de forma controlada
    @property
    def precio(self) -> float:
        return self.__precio

    # Setter: valida las reglas de negocio antes de modificar el dato
    @precio.setter
    def precio(self, nuevo_precio: float):
        if nuevo_precio <= 0:
            raise ValueError("El precio de un producto debe ser estrictamente mayor a 0.")
        self.__precio = nuevo_precio

# Pruebas de encapsulamiento
try:
    prod = Producto("Licencia SaaS Pro", 99.99)
    print(f"Producto creado: {prod.nombre} - Precio: ${prod.precio}")
    
    # Intentamos cambiar a un precio no permitido
    prod.precio = -10.0
except ValueError as e:
    print(f"Error detectado y controlado: {e}")


## 3. Herencia y Polimorfismo

- **Herencia**: Permite que una clase hija adquiera los atributos y métodos de una clase padre, facilitando la reutilización del código.
- **Polimorfismo**: Permite que diferentes clases expongan el mismo nombre de método pero lo implementen de formas distintas (comportamientos especializados).


In [ ]:
# Clase Padre
class Suscripcion:
    def __init__(self, precio_base: float):
        self.precio_base = precio_base

    def calcular_tarifa(self) -> float:
        # Tarifa estándar
        return self.precio_base

# Clase Hija 1: Hereda de Suscripcion
class SuscripcionAnual(Suscripcion):
    def calcular_tarifa(self) -> float:
        # Polimorfismo: aplica un 20% de descuento por pago adelantado
        return (self.precio_base * 12) * 0.8

# Clase Hija 2: Hereda de Suscripcion
class SuscripcionFamiliar(Suscripcion):
    def __init__(self, precio_base: float, miembros: int):
        # Llamamos al constructor del padre
        super().__init__(precio_base)
        self.miembros = miembros

    def calcular_tarifa(self) -> float:
        # Costo base multiplicado por miembros, con 10% de descuento
        costo_total = self.precio_base * self.miembros
        return costo_total * 0.9

# Ejecutando comportamiento polimórfico
planes = [
    Suscripcion(50.0),                     # Mensual simple
    SuscripcionAnual(50.0),                # Anual con descuento
    SuscripcionFamiliar(50.0, 4)           # Familiar con 4 personas
]

for idx, plan in enumerate(planes, 1):
    print(f"Plan {idx} - Costo final: ${plan.calcular_tarifa():.2f}")


## 4. Modelado Completo de un Flujo de E-commerce

A continuación veremos un ejemplo integrado donde múltiples clases colaboran para modelar un flujo de checkout completo de comercio electrónico.


In [ ]:
class ItemCarrito:
    def __init__(self, producto: Producto, cantidad: int):
        self.producto = producto
        self.cantidad = cantidad

    @property
    def subtotal(self) -> float:
        return self.producto.precio * self.cantidad

class Carrito:
    def __init__(self):
        self.items = []

    def agregar_item(self, producto: Producto, cantidad: int = 1):
        # Buscamos si ya está el producto
        for item in self.items:
            if item.producto.nombre == producto.nombre:
                item.cantidad += cantidad
                return
        self.items.append(ItemCarrito(producto, cantidad))

    @property
    def total(self) -> float:
        return sum(item.subtotal for item in self.items)

    def mostrar_detalle(self):
        print("=== Detalle del Carrito ===")
        for item in self.items:
            print(f" - {item.producto.nombre} x{item.cantidad}: ${item.subtotal:.2f}")
        print(f"Total: ${self.total:.2f}")

# Simulación de compra
prod1 = Producto("Suscripción CRM", 45.0)
prod2 = Producto("Módulo Analítica", 25.0)

carrito = Carrito()
carrito.agregar_item(prod1, 1)
carrito.agregar_item(prod2, 2)
carrito.agregar_item(prod1, 1) # Aumenta cantidad de CRM a 2

carrito.mostrar_detalle()


---

## Desafío Práctico (Trabajo Práctico 2)

**Consigna de Negocio (Modelado de Transacciones):**
Un e-commerce de Negocios Digitales necesita implementar una capa de transacciones con soporte para distintos medios de pago.

1. Diseña una clase `Transaccion` básica que contenga:
   - Atributo privado `__monto` con su respectiva `@property` y validación de que sea mayor a 0.
   - Método `procesar_pago()` que retorne un string `"Procesando transacción general."`
2. Diseña dos clases hijas que hereden de `Transaccion`:
   - `TransaccionMercadoPago`: Debe inicializarse con el `email_pagador` y sobrescribir `procesar_pago()` para que devuelva:
     `"Cobrando $[monto] a [email_pagador] mediante Mercado Pago (QR/Dinero en cuenta)."`
   - `TransaccionStripe`: Debe inicializarse con la `moneda` (ej. `"USD"`, `"EUR"`) y sobrescribir `procesar_pago()` para que devuelva:
     `"Procesando cobro internacional de [moneda] $[monto] con tarjeta de crédito vía Stripe."`
3. Implementa una clase `TransaccionFactory` con un método estático `@staticmethod` llamado `crear_transaccion(canal: str, monto: float, **kwargs)` que retorne una instancia de la transacción adecuada según el canal (`"mercadopago"` o `"stripe"`). Lanza un `ValueError` si el canal no está soportado.
4. Escribe un flujo de prueba donde simules el pago de un carrito de compras de `$150.00` usando Stripe (en dólares) y un pago de `$4500.00` usando Mercado Pago, imprimiendo el resultado de `procesar_pago()` para cada transacción generada por la fábrica.

Implementa tu código a continuación.


In [ ]:
# Escribe la resolución aquí
# 1. Definir clase Transaccion
# ...

# 2. Definir clases hijas TransaccionMercadoPago y TransaccionStripe
# ...

# 3. Definir TransaccionFactory
# ...

# 4. Flujo de prueba
# ...
